In [1]:
# ============================================================
# Notebook : 06_gate_ablation.ipynb
# Purpose  : Demonstrate the UTILITY of the framework by removing each
#            gate in turn and quantifying the flawed recommendation that
#            is then admitted. The framework's claim - that its value lies
#            in what it excludes and keeps traceable, not in what it passes
#            - is tested here: each row shows a concrete erroneous output
#            that a conventional, gate-free pipeline would let through.
#              Gate 1 (sample eligibility)  : facility contamination of the
#                                             high-risk threshold.
#              Gate 3 (causal stability)    : non-reproducible precursors
#                                             asserted as stable.
#              Gate 4a (reliability)        : near-zero-precision categories
#                                             auto-recommended.
#              Gate 4b (causal constraint)  : counterfactuals editing
#                                             off-path (non-causal) variables.
# Design   : Single code cell. No absolute path is printed.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "axes.unicode_minus": False,
    "savefig.dpi": 600, "figure.dpi": 120, "axes.edgecolor": "0.15", "grid.color": "0.85",
})
GRAY = ["0.15", "0.40", "0.60", "0.75", "0.88"]


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"

home = pd.read_pickle(DATA / "analysis_home.pkl")
facility = pd.read_pickle(DATA / "facility_track.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
repro = pd.read_pickle(DATA / "reproduction.pkl")
perf = pd.read_csv(TAB / "table_model_performance.csv")
cross_all = pd.read_csv(TAB / "table_all_cross_links.csv")
cross_stable = pd.read_csv(TAB / "table_stable_cross_links.csv")
bypass = pd.read_csv(TAB / "table_dice_bypass.csv")
ABN = meta["abnormal_categories"]

results = []

# ============================================================
# Gate 1 OFF: keep the facility track in the sample.
#   A conventional pipeline that does not separate structurally distinct
#   observation environments lets the facility households set the top of
#   the household risk distribution.
# ============================================================
home_hh = home.groupby("HH_ID")["sev_danger"].mean()
fac_hh = facility.groupby("HH_ID")["sev_danger"].mean()
home_p90 = float(np.percentile(home_hh, 90))
fac_mean = float(fac_hh.mean())
# top-decile "very high risk" band defined from home only
band_home = set(home_hh[home_hh >= home_p90].index)
# when facility is pooled in, which households occupy the top-2 risk ranks
pooled = pd.concat([home_hh, fac_hh]).sort_values(ascending=False)
top2_pooled = list(pooled.head(2).index)
infl = fac_mean / home_p90
results.append({
    "gate": "1 Sample eligibility",
    "ablation": "facility track retained in sample",
    "flawed_output_admitted": "facility households set the high-risk ceiling; "
                              "home risk band mis-scaled",
    "magnitude": f"facility danger rate {fac_mean:.3f} = {infl:.2f}x the home 90th pct "
                 f"({home_p90:.3f}); top-2 pooled risk ranks both facility {top2_pooled}",
    "score": float(min(infl / 2.0, 1.0)),  # normalized for the figure
})

# ============================================================
# Gate 3 OFF: assert every discovered cross-link as a stable precursor.
# ============================================================
n_all = len(cross_all)
n_stable = len(cross_stable)
non_repro_links = n_all - n_stable
cat_non_repro = [c for c in ABN if repro.get(c) in ("not_reproduced", "autoregressive_only")]
results.append({
    "gate": "3 Causal stability",
    "ablation": "no reproducibility / stability filter",
    "flawed_output_admitted": "non-reproducible lag-1 links asserted as causal precursors",
    "magnitude": f"{non_repro_links} of {n_all} cross-links ({non_repro_links/n_all:.0%}) "
                 f"would be asserted though not recurring; "
                 f"{len(cat_non_repro)} categories lack a reproduced precursor {cat_non_repro}",
    "score": float(non_repro_links / n_all),
})

# ============================================================
# Gate 4a OFF: skip the dual reliability criterion.
#   Categories with high AUC but near-zero precision would be auto-used.
# ============================================================
perf_idx = perf.set_index("category")
held = [c for c in ABN if c in perf_idx.index and not bool(perf_idx.loc[c, "passes_gate"])]
worst_lines = []
worst_fp = 0.0
for c in held:
    p = perf_idx.loc[c, "precision"]
    a = perf_idx.loc[c, "auc"]
    if pd.notna(p):
        fp_rate = 1.0 - float(p)
        worst_fp = max(worst_fp, fp_rate)
        worst_lines.append(f"{c}: AUC {float(a):.3f}, precision {float(p):.3f} "
                           f"-> {fp_rate:.1%} of auto-recommendations would be false")
results.append({
    "gate": "4a Model reliability",
    "ablation": "AUC-only acceptance (no precision / positive-count floor)",
    "flawed_output_admitted": "near-zero-precision categories auto-recommended",
    "magnitude": "; ".join(worst_lines),
    "score": float(worst_fp),
})

# ============================================================
# Gate 4b OFF: unconstrained counterfactuals (no causal-path restriction).
# ============================================================
mean_bypass = float(bypass["bypass_rate_offpath"].mean())
max_bypass = float(bypass["bypass_rate_offpath"].max())
worst_cat = bypass.sort_values("bypass_rate_offpath", ascending=False).iloc[0]
results.append({
    "gate": "4b Causal constraint",
    "ablation": "unconstrained DiCE (edit any variable)",
    "flawed_output_admitted": "recommendations editing off-path (non-causal) variables",
    "magnitude": f"mean off-path edit rate {mean_bypass:.1%} across categories; "
                 f"worst {worst_cat['category']} {float(worst_cat['bypass_rate_offpath']):.1%} "
                 f"(top off-path edit: {worst_cat['most_frequent_offpath_edit']})",
    "score": mean_bypass,
})

ablation = pd.DataFrame(results)
ablation.to_csv(TAB / "table_gate_ablation.csv", index=False, encoding="utf-8-sig")

# ============================================================
# Figure 11. Share of flawed output admitted when each gate is removed.
# ============================================================
fig, ax = plt.subplots(figsize=(7.0, 3.4))
lab = ablation["gate"]
ax.barh(range(len(ablation)), ablation["score"], color=GRAY[1], edgecolor="black", linewidth=0.6)
ax.set_yticks(range(len(ablation)))
ax.set_yticklabels(lab)
ax.invert_yaxis()
ax.set_xlim(0, 1.0)
ax.set_xlabel("Normalized severity of admitted flaw (0-1)")
save_fig(fig, "fig11_gate_ablation")

# ---- Console summary ----
print("=== Gate ablation: what each removed gate admits ===")
for r in results:
    print(f"\n[{r['gate']}]  remove -> {r['ablation']}")
    print(f"   admits : {r['flawed_output_admitted']}")
    print(f"   extent : {r['magnitude']}")
print()
print("Figure saved : fig11_gate_ablation  (png + pdf, 600 dpi)")
print("Table saved  : table_gate_ablation")


=== Gate ablation: what each removed gate admits ===

[1 Sample eligibility]  remove -> facility track retained in sample
   admits : facility households set the high-risk ceiling; home risk band mis-scaled
   extent : facility danger rate 0.101 = 1.94x the home 90th pct (0.052); top-2 pooled risk ranks both facility ['A0030', 'A0029']

[3 Causal stability]  remove -> no reproducibility / stability filter
   admits : non-reproducible lag-1 links asserted as causal precursors
   extent : 33 of 42 cross-links (79%) would be asserted though not recurring; 0 categories lack a reproduced precursor []

[4a Model reliability]  remove -> AUC-only acceptance (no precision / positive-count floor)
   admits : near-zero-precision categories auto-recommended
   extent : SP: AUC 0.997, precision 0.377 -> 62.3% of auto-recommendations would be false; ER: AUC 0.750, precision 0.000 -> 100.0% of auto-recommendations would be false

[4b Causal constraint]  remove -> unconstrained DiCE (edit any variable